# 🎬 Autonomous Anime Studio: Dual-Engine AI Director (Google Colab & Kaggle)
**Autonomous Vision-Language Editor & YouTube Studio Scheduler**

### Complete Workflow Architecture:
1. **Strict Category Selection**: Preserves channel format rules (Group A: 5-clip ranking, Group B: Rotate Phone AMV, Group C: Micro-moment).
2. **Sound-First Virality Radar**: Discovers trending audio from Instagram Reels & TikTok.
3. **Candidate Scouting & Scoring**: Gathers high-velocity clips and scores action metrics.
4. **Local Vision-LLM Analysis (`Qwen2-VL-2B`)**: Evaluates frame-by-frame quality, checks for non-anime pollution, and pinpoints the exact climax cut window.
5. **Audio-Visual Rendering**: Composites 1080x1920 with minimal glassmorphism HUD, channel watermark, and creator credit.
6. **Self-Healing Supervisor Loop**: Audits the finished video for black frames, silence, or formatting issues, automatically fixes any defect, and rechecks.
7. **SEO Optimization & Hashtag Hijacking**: Generates high-CTR title, fair-use disclaimers, and stacks viral unrelated tags to maximize YouTube Shorts feed distribution.
8. **Dual-Engine Quota Failover**: Switch between Colab (T4 GPU) and Kaggle (T4/P100 GPU) when either platform exhausts its free weekly GPU quota.

In [ ]:
# [Cell 1] Environment Detection & Fast Dependency Setup
import os, sys, subprocess

is_colab = "google.colab" in sys.modules or os.path.exists("/content")
is_kaggle = os.path.exists("/kaggle")

print(f"⚡ Compute Host: {'Google Colab' if is_colab else ('Kaggle Notebook' if is_kaggle else 'Local Engine')}")

# Install dependencies silently
!apt-get update -qq && apt-get install -y -qq ffmpeg fonts-dejavu-core fonts-freefont-ttf > /dev/null
!pip install -q yt-dlp pillow torch torchvision transformers accelerate google-api-python-client google-auth-oauthlib

# Clone or pull latest codebase
if not os.path.exists("anime-AUTO"):
    !git clone https://github.com/sp25126/anime-AUTO.git
    %cd anime-AUTO
else:
    %cd anime-AUTO
    !git pull origin main


In [ ]:
# [Cell 2] Restore YouTube Token from Cloud Secrets (Zero Laptop Dependency)
import os, json

os.makedirs("config", exist_ok=True)
token_found = False

if is_colab:
    try:
        from google.colab import userdata
        token_val = userdata.get("YOUTUBE_TOKEN_JSON")
        with open("config/youtube_token.json", "w") as f:
            f.write(token_val)
        print("✓ Restored YouTube Token from Colab Secrets!")
        token_found = True
    except Exception:
        print("Colab Secret 'YOUTUBE_TOKEN_JSON' not found. Checking Drive...")
        try:
            from google.colab import drive
            drive.mount("/content/drive", force_remount=False)
            p = "/content/drive/MyDrive/youtube_token.json"
            if os.path.exists(p):
                import shutil
                shutil.copy2(p, "config/youtube_token.json")
                print("✓ Restored YouTube Token from Google Drive!")
                token_found = True
        except Exception as e:
            print(f"Drive notice: {e}")

elif is_kaggle:
    try:
        from kaggle_secrets import UserSecretsClient
        token_val = UserSecretsClient().get_secret("YOUTUBE_TOKEN_JSON")
        with open("config/youtube_token.json", "w") as f:
            f.write(token_val)
        print("✓ Restored YouTube Token from Kaggle Secrets!")
        token_found = True
    except Exception as e:
        print(f"Kaggle secret notice: {e}")

if not token_found:
    print("⚠️ Running in LOCAL PREVIEW mode (video will render & save locally without upload).")


In [ ]:
# [Cell 3] Run Autonomous Production Cycle with Local Vision Director & Self-Healing
import sys, os
sys.path.append("scripts/engine")

from autonomous_channel_manager import AutonomousChannelManager
from local_vision_director import LocalVisionAIDirector

# 1. Load Local Small Vision Model onto GPU (Qwen2-VL-2B in FP16/4-bit)
director = LocalVisionAIDirector(use_gpu=True)
director.load_local_model()

# 2. Initialize Master Manager
manager = AutonomousChannelManager()
is_upload = os.path.exists("config/youtube_token.json")

print(f"\n🚀 Starting Autonomous Cycle | Mode: {'LIVE YOUTUBE UPLOAD' if is_upload else 'LOCAL PREVIEW'}")
success = manager.run_zero_touch_cycle(local_only=not is_upload)

# 3. Self-Healing Post-Render Audit
if success:
    print("\n==================================================================")
    print("🎉 PRODUCTION SUCCEEDED!")
    print("Check the output folder or YouTube Studio for your scheduled release.")
    print("==================================================================")
else:
    print("\n❌ Production cycle encountered an error. Check supervisor log above.")
